# exploracao_gold_dec_fec

Caça de problemas no `gold_dec_fec` e no que ele usa: `silver.tb_ocorrencias_rede` (silver_rede_atendimento) e `gold.dim_alimentador`, `gold.dim_uc` e `gold.dim_mes` (gold_base). **Não grava nada.**

- **Rode depois do Job** (precisa das Silver e das Gold `gold_base`, `gold_dec_fec` e `gold_kpis`).
- Cada seção responde a um item da lista de possíveis problemas. O número entre parênteses é o item.

| Seção | Pergunta | Item |
|---|---|---|
| 1 | Quanto cada tipo de ocorrência pesa no DEC? | #1 |
| 2 | Como fica o DEC/FEC por período com e sem os tipos que não são falta de energia? | #1 |
| 3 | Quais dias seriam "dia crítico" (média + 3 desvios)? | #2 |
| 4 | Do que é feito julho de 2025? | #2 |
| 5 | O ranking por DEC destaca alimentadores pequenos? | #3 |
| 6 | Quantas horas a sobreposição conta em dobro? | #4 |
| 7 | Quanto o DEC muda com o Cc por mês em vez do Cc fixo? | #5 |
| 8 | Há interrupção antes da ligação da UC? | #6 |
| 9 | Quantas interrupções viram o mês? | #7 |
| 10 | O DEC/FEC do `gold_dec_fec` bate com o do `gold_kpis`? | #10 |
| 11 | As flags da Silver que o `gold_dec_fec` não checa estão zeradas? | #8 e #9 |

In [0]:
%run ../bronze-to-silver/funcoes_comuns

In [0]:
dbutils.widgets.text("catalog", "workspace")
CATALOGO = dbutils.widgets.get("catalog")

OC = f"{CATALOGO}.silver.tb_ocorrencias_rede"
G = f"{CATALOGO}.gold"
N_UCS = spark.table(f"{G}.dim_uc").count()   # Cc global (45.000)
sql = lambda q: display(spark.sql(q))
print(f"ocorrências: {spark.table(OC).count():,} | UCs: {N_UCS:,}")

## 1. Peso de cada tipo de ocorrência no DEC (#1)
Hoje todos os tipos entram como interrupção. OSCILACAO é variação de tensão, não falta de energia. **Olhe:** `pct_dec` dos tipos que não são FALTA DE ENERGIA.

In [0]:
sql(f'''
SELECT tipo, COUNT(*) AS interrupcoes, ROUND(SUM(duracao_min) / 60, 0) AS horas,
       ROUND(100 * SUM(duracao_min) / SUM(SUM(duracao_min)) OVER (), 1) AS pct_dec
FROM {OC} GROUP BY tipo ORDER BY interrupcoes DESC
''')

## 2. DEC/FEC global por período, por tipo considerado (#1)
**Olhe:** quanto o DEC e o FEC de P1 e P2 mudam se só FALTA DE ENERGIA contar, ou se só a OSCILACAO sair.

In [0]:
sql(f'''
SELECT d.periodo,
       ROUND(SUM(o.duracao_min) / 60 / {N_UCS}, 4) AS dec_todos,
       ROUND(SUM(CASE WHEN o.tipo = 'FALTA DE ENERGIA' THEN o.duracao_min END) / 60 / {N_UCS}, 4) AS dec_so_falta,
       ROUND(SUM(CASE WHEN o.tipo <> 'OSCILACAO' THEN o.duracao_min END) / 60 / {N_UCS}, 4) AS dec_sem_oscilacao,
       ROUND(COUNT(*) / {N_UCS}, 4) AS fec_todos,
       ROUND(COUNT_IF(o.tipo = 'FALTA DE ENERGIA') / {N_UCS}, 4) AS fec_so_falta,
       ROUND(COUNT_IF(o.tipo <> 'OSCILACAO') / {N_UCS}, 4) AS fec_sem_oscilacao
FROM {OC} o
JOIN {G}.dim_mes d ON d.mes_referencia = date_format(o.data_abertura, 'yyyy-MM')
WHERE d.periodo IS NOT NULL
GROUP BY d.periodo ORDER BY d.periodo
''')

## 3. Dias candidatos a "dia crítico" (#2)
Critério do tipo do PRODIST: dia com interrupções acima da média + 3 desvios. **Olhe:** se os dias críticos se concentram em julho de 2025 e quantas horas um expurgo tiraria.

In [0]:
sql(f'''
WITH dia AS (SELECT CAST(data_abertura AS DATE) AS dia, COUNT(*) AS n, SUM(duracao_min) / 60 AS horas
             FROM {OC} GROUP BY 1),
lim AS (SELECT AVG(n) + 3 * STDDEV(n) AS limite FROM dia)
SELECT date_format(dia, 'yyyy-MM') AS mes, COUNT(*) AS dias_criticos, SUM(n) AS interrupcoes,
       ROUND(SUM(horas)) AS horas, ROUND(MAX(limite)) AS limite_por_dia
FROM dia CROSS JOIN lim WHERE n > limite
GROUP BY 1 ORDER BY 1
''')

## 4. Composição de julho de 2025 (#2)
**Olhe:** se o mês é um evento só (uma causa e um tipo) ou uma mistura.

In [0]:
sql(f'''
SELECT causa, tipo, COUNT(*) AS n, ROUND(SUM(duracao_min) / 60) AS horas
FROM {OC} WHERE date_format(data_abertura, 'yyyy-MM') = '2025-07'
GROUP BY ALL ORDER BY n DESC
''')

## 5. Ranking de P2: DEC x horas x UCs afetadas (#3)
**Olhe:** alimentadores pequenos no topo do DEC com poucas horas absolutas; compare `pct_ucs_afetadas` e `horas`.

In [0]:
sql(f'''
SELECT a.alimentador, a.municipio, a.ucs_atendidas,
       COUNT(*) AS interrupcoes, COUNT(DISTINCT o.id_uc) AS ucs_afetadas,
       ROUND(COUNT(DISTINCT o.id_uc) / a.ucs_atendidas, 2) AS pct_ucs_afetadas,
       ROUND(SUM(o.duracao_min) / 60, 1) AS horas,
       ROUND(SUM(o.duracao_min) / 60 / a.ucs_atendidas, 2) AS dec_p2
FROM {OC} o JOIN {G}.dim_alimentador a USING (alimentador)
WHERE date_format(o.data_abertura, 'yyyy-MM') BETWEEN '2025-07' AND '2026-06'
GROUP BY ALL ORDER BY dec_p2 DESC LIMIT 20
''')

## 6. Sobreposição: horas contadas em dobro (#4)
Interrupção que começa antes do fim de outra da mesma UC. **Olhe:** o total de `horas_em_dobro` e os meses em que se concentra.

In [0]:
sql(f'''
WITH o AS (
  SELECT *, MAX(data_encerramento) OVER (PARTITION BY id_uc ORDER BY data_abertura, id_ocorrencia
                                         ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS fim_anterior
  FROM {OC})
SELECT date_format(data_abertura, 'yyyy-MM') AS mes, COUNT(*) AS sobrepostas,
       ROUND(SUM(timestampdiff(MINUTE, data_abertura, LEAST(data_encerramento, fim_anterior))) / 60, 1) AS horas_em_dobro
FROM o WHERE data_abertura < fim_anterior
GROUP BY 1 ORDER BY horas_em_dobro DESC
''')

## 7. Cc fixo x Cc por mês (#5)
Hoje o Cc é o cadastro atual. Com o Cc por mês, só contam as UCs ligadas até o fim do mês. **Olhe:** a diferença entre `dec_cc_fixo` e `dec_cc_mes`.

In [0]:
sql(f'''
WITH cc_mes AS (
  SELECT u.alimentador, m.mes_referencia, COUNT(*) AS cc_mes
  FROM {G}.dim_uc u CROSS JOIN {G}.dim_mes m
  WHERE u.data_ligacao <= last_day(to_date(m.mes_referencia || '-01'))
  GROUP BY ALL)
SELECT g.alimentador, g.mes_referencia, g.ucs_atendidas AS cc_fixo, c.cc_mes,
       g.dec AS dec_cc_fixo, ROUND(g.horas_interrupcao / c.cc_mes, 4) AS dec_cc_mes
FROM {G}.agg_dec_fec_mensal g JOIN cc_mes c USING (alimentador, mes_referencia)
WHERE g.ucs_atendidas <> c.cc_mes
ORDER BY g.ucs_atendidas - c.cc_mes DESC LIMIT 30
''')

## 8. Interrupções antes da ligação da UC (#6)
A UC ainda não existia quando a interrupção foi aberta. **Olhe:** quantas são e em que meses.

In [0]:
sql(f'''
SELECT date_format(o.data_abertura, 'yyyy-MM') AS mes, COUNT(*) AS n, ROUND(SUM(o.duracao_min) / 60, 1) AS horas
FROM {OC} o JOIN {G}.dim_uc u USING (id_uc)
WHERE o.data_abertura < u.data_ligacao
GROUP BY 1 ORDER BY 1
''')

## 9. Interrupções que viram o mês (#7)
Hoje a duração inteira fica no mês de abertura. **Olhe:** quantas horas iriam para o mês seguinte.

In [0]:
sql(f'''
SELECT date_format(data_abertura, 'yyyy-MM') AS mes_abertura, COUNT(*) AS n,
       ROUND(SUM(duracao_min) / 60) AS horas_no_mes_de_abertura
FROM {OC}
WHERE date_format(data_abertura, 'yyyy-MM') <> date_format(data_encerramento, 'yyyy-MM')
GROUP BY 1 ORDER BY n DESC
''')

## 10. gold_dec_fec x gold_kpis (#10)
Os dois devem dar o mesmo DEC/FEC global. **Olhe:** se aparece alguma linha (diferença); vazio = batem.

In [0]:
sql(f'''
WITH dec_fec AS (
  SELECT mes_referencia, ROUND(SUM(horas_interrupcao) / {N_UCS}, 4) AS dec_dec_fec,
         ROUND(SUM(interrupcoes) / {N_UCS}, 4) AS fec_dec_fec
  FROM {G}.agg_dec_fec_mensal GROUP BY 1)
SELECT k.mes_referencia, k.dec AS dec_kpis, d.dec_dec_fec, k.fec AS fec_kpis, d.fec_dec_fec
FROM {G}.agg_kpis_mensal k JOIN dec_fec d USING (mes_referencia)
WHERE k.dec <> d.dec_dec_fec OR k.fec <> d.fec_dec_fec
ORDER BY 1
''')

## 11. Flags da Silver que o gold_dec_fec não checa (#8 e #9)
**Esperado:** 0 nas três flags. `exatamente_3_min` mostra o que muda se a regra virar `>= 3`.

In [0]:
sql(f'''
SELECT COUNT_IF(flag_alimentador_inconsistente) AS alimentador_inconsistente,
       COUNT_IF(flag_duracao_inconsistente) AS duracao_inconsistente,
       COUNT_IF(flag_encerramento_antes_abertura) AS encerramento_antes_abertura,
       COUNT_IF(duracao_min = 3) AS exatamente_3_min,
       MIN(duracao_min) AS menor_duracao
FROM {OC}
''')